# CATSA Binary Stress Classification with Mamba2 + LOSO CV (ACC Only)

## Overview
- **모델**: Mamba2 (State Space Model) with single ACC branch
- **학습 데이터**: CATSA dataset (53 subjects)
- **테스트 데이터**: EmpaticaE4Stress dataset (29 subjects)
- **학습 방식**: Leave-One-Subject-Out (LOSO) cross-validation
- **분류 대상**: Binary (Non-Stress=0 vs Stress=1)
- **입력 신호**: **ACC만 사용** (EDA, TEMP, HR, HRV 제외)

## Label Mapping
- `0`: Non-Stress → `Baseline`
- `1`: Stress → `Logic`, `Sudoku`, `Stroop`

## Input Signals
- **ACC (32Hz)**: `(B, 3, 1920)` for 60-second windows
- Direct input to model (downsampled to 240 timepoints for Mamba2)

## Architecture
- ACC input: (B, 3, 1920) → downsampled to (B, 3, 240)
- **Mamba2**: d_model=3 (input directly), d_state=64, expand=2
- Head: Mean pooling → FC(2)

In [1]:
import math
import random
import sys
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import find_peaks, butter, filtfilt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import Dataset, DataLoader

# Mamba2 import
sys.path.insert(0, '/home/binghin2/Model/mamba')
from mamba_ssm.modules.mamba2 import Mamba2

# ===================== Configuration =====================
DATASET_ROOT = Path('/home/binghin2/Myproject/Dataset/CATSA')
EMPATICA_ROOT = Path('/home/binghin2/Myproject/Dataset/EmpaticaE4Stress/Subjects')
OUTPUT_DIR = Path('/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ACC/Mamba2/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Task mapping
TASK_TO_CLASS = {
    'Baseline': 0,
    'Logic': 1,
    'Sudoku': 1,
    'Stroop': 1,
}
CLASS_NAMES = ['NonStress', 'Stress']

# Window parameters
WINDOW_SEC = 60
TRAIN_STRIDE_SEC = 20
STRIDE_SEC_EVAL = 60

# Sampling rates (ACC only)
FS_ACC = 32

# Expected lengths
LEN_ACC = WINDOW_SEC * FS_ACC   # 1920
LEN_ACC_MAMBA = 240  # Downsampled to 240 timepoints for Mamba2

# Training parameters
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

# Model hyperparameters
DROPOUT = 0.4
EARLY_STOP_PATIENCE = 8
EARLY_STOP_F1_MIN_DELTA = 1e-4

# Mamba2 config (simplified for ACC only)
MAMBA_D_MODEL = 3  # 3-axis accelerometer directly
MAMBA_D_STATE = 64
MAMBA_EXPAND = 2

# Imbalance controls
USE_CLASS_WEIGHTS = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('Device:', DEVICE)
print('Dataset root:', DATASET_ROOT)
print('Input signal: ACC only (3-axis, 32Hz)')
print('Mamba2 config:', {'d_model': MAMBA_D_MODEL, 'd_state': MAMBA_D_STATE, 'expand': MAMBA_EXPAND})
print('Early stopping patience:', EARLY_STOP_PATIENCE)

ModuleNotFoundError: No module named 'selective_scan_cuda'

In [ ]:
# ===================== ACC-Only Data Loading =====================

def read_csv_array(path: Path) -> np.ndarray:
    """Read CSV file and return as float32 array."""
    return pd.read_csv(path).values.astype(np.float32)

def load_acc_only(subject_dir: Path, task: str) -> np.ndarray:
    """Load ACC data for a task. Returns (T, 3) array."""
    task_dir = subject_dir / task
    acc_path = task_dir / 'ACC.csv'   
    
    if not acc_path.exists():
        return None
    
    acc = read_csv_array(acc_path)
    
    # Ensure 3-axis format
    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)
    if acc.shape[1] < 3:
        acc = np.tile(acc, (1, 3))[:, :3]
    else:
        acc = acc[:, :3]
    
    return acc.astype(np.float32)

def downsample_acc(acc: np.ndarray, fs_orig: int = 32, fs_target: int = 4) -> np.ndarray:
    """Downsample ACC from 32Hz to 4Hz using simple averaging."""
    ratio = fs_orig // fs_target
    n_samples = len(acc) // ratio
    resampled = np.zeros((n_samples, acc.shape[1]), dtype=np.float32)
    
    for i in range(n_samples):
        resampled[i] = acc[i*ratio:(i+1)*ratio].mean(axis=0)
    
    return resampled

def compute_subject_baseline_stats(subject_dir: Path) -> dict:
    """Compute per-subject normalization stats from Baseline task (ACC only)."""
    acc = load_acc_only(subject_dir, 'Baseline')
    if acc is None:
        return None
    
    acc_mean = acc.mean(axis=0, keepdims=True)
    acc_std = acc.std(axis=0, keepdims=True) + 1e-6
    
    return {
        'acc_mean': acc_mean.astype(np.float32),
        'acc_std': acc_std.astype(np.float32),
    }

def apply_subject_baseline_norm(acc: np.ndarray, stats: dict) -> np.ndarray:
    """Apply subject-wise baseline normalization."""
    acc_norm = (acc - stats['acc_mean']) / stats['acc_std']
    return acc_norm.astype(np.float32)

def create_windows_acc(acc: np.ndarray, label: int, stride_sec: int) -> list:
    """Create sliding windows from ACC data only."""
    stride_samples = stride_sec * FS_ACC
    window_samples = LEN_ACC
    
    n_samples = len(acc)
    windows = []
    start = 0
    
    while start + window_samples <= n_samples:
        acc_win = acc[start:start + window_samples].T.copy()  # (3, 1920)
        
        windows.append({
            'acc': acc_win.astype(np.float32),
            'y': int(label),
        })
        start += stride_samples
    
    return windows

def list_subjects(root: Path) -> list:
    """List all subject directories."""
    return sorted([p.name for p in root.glob('Sub*') if p.is_dir()])

def load_loso_fold(test_subject: str, all_subjects: list, seed: int = 42):
    """Split subjects for LOSO: test=specified, val=random, train=rest."""
    remaining = [s for s in all_subjects if s != test_subject]
    rng = np.random.RandomState(seed)
    val_subject = rng.choice(remaining)
    train_subjects = [s for s in remaining if s != val_subject]
    
    return train_subjects, [val_subject], [test_subject]

def build_split_windows(subjects: list, is_train: bool = True) -> list:
    """Build windows for a list of subjects."""
    all_windows = []
    skipped_count = 0
    
    for subject in subjects:
        sdir = DATASET_ROOT / subject
        stats = compute_subject_baseline_stats(sdir)
        
        if stats is None:
            skipped_count += 1
            continue
        
        for task, cls in TASK_TO_CLASS.items():
            acc = load_acc_only(sdir, task)
            if acc is None:
                continue
            
            # Apply normalization
            acc_norm = apply_subject_baseline_norm(acc, stats)
            
            # Create windows
            stride = TRAIN_STRIDE_SEC if is_train else STRIDE_SEC_EVAL
            windows = create_windows_acc(acc_norm, cls, stride_sec=stride)
            all_windows.extend(windows)
    
    if skipped_count > 0:
        print(f'  Skipped {skipped_count} subjects (missing Baseline)')
    
    return all_windows

# Load subject list
all_subjects = list_subjects(DATASET_ROOT)
print(f'Loaded {len(all_subjects)} CATSA subjects')

print('Data loading functions ready.')

In [ ]:
# ===================== Model Definition =====================

class Mamba2Stress(nn.Module):
    """
    Mamba2-based stress classifier using ACC data only.
    
    Input: ACC (B, 3, 1920) at 32Hz
    Output: Logits (B, 2)
    """
    def __init__(self, n_classes=2, dropout=0.4, d_model=3, d_state=64, expand=2):
        super().__init__()
        self.d_model = d_model
        self.dropout = dropout
        
        # Downsample ACC: (B, 3, 1920) -> (B, 3, 240)
        # Using AdaptiveAvgPool1d for efficient downsampling
        self.downsample = nn.AdaptiveAvgPool1d(LEN_ACC_MAMBA)
        
        # Mamba2 SSM block
        self.mamba = Mamba2(
            d_model=d_model,
            d_state=d_state,
            expand=expand,
            headdim=64,
            d_conv=4,
            conv_init=None,
            dt_min=0.001,
            dt_max=0.1,
            dt_init_floor=1e-4,
            bias=False,
            conv_bias=True,
            chunk_size=256,
            use_mem_eff_path=True,
        )
        
        # Classification head
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )
    
    def forward(self, acc):
        """
        Forward pass.
        
        Args:
            acc: (B, 3, 1920) at 32Hz
        
        Returns:
            logits: (B, 2)
        """
        # Downsample to 240 timepoints
        x = self.downsample(acc)  # (B, 3, 240)
        
        # Permute to (B, 240, 3) for Mamba2
        x = x.permute(0, 2, 1).contiguous()  # (B, 240, 3)
        
        # Apply Mamba2
        x = self.mamba(x)  # (B, 240, 3)
        
        # Global average pooling
        x = x.mean(dim=1)  # (B, 3)
        
        # Normalize and classify
        x = self.norm(x)
        logits = self.head(x)  # (B, 2)
        
        return logits


# Test forward pass
def test_model():
    model = Mamba2Stress(n_classes=2, dropout=0.4, d_model=3, d_state=64, expand=2).to(DEVICE)
    acc_dummy = torch.randn(2, 3, LEN_ACC, device=DEVICE)
    logits = model(acc_dummy)
    print(f'✓ Model test passed. Input: {tuple(acc_dummy.shape)}, Output: {tuple(logits.shape)}')
    return model

model = test_model()

In [ ]:
# ===================== Dataset Class =====================

class ACCDataset(Dataset):
    """PyTorch dataset for ACC-only windows."""
    
    def __init__(self, windows):
        self.windows = windows
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        w = self.windows[idx]
        return (
            torch.tensor(w['acc'], dtype=torch.float32),
            torch.tensor(w['y'], dtype=torch.long),
        )


# ===================== Training Utilities =====================

def class_weights_from_windows(windows: list, n_classes: int) -> torch.Tensor:
    """Compute class weights from window labels."""
    y = np.array([w['y'] for w in windows], dtype=np.int64)
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    total = counts.sum()
    weights = total / (n_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32)


def eval_model(model, loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    all_pred = []
    all_true = []
    
    with torch.no_grad():
        for acc, y in loader:
            acc = acc.to(device)
            y = y.to(device)
            
            logits = model(acc)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * y.size(0)
            pred = logits.argmax(dim=1)
            all_pred.extend(pred.cpu().numpy().tolist())
            all_true.extend(y.cpu().numpy().tolist())
    
    n = max(len(all_true), 1)
    avg_loss = total_loss / n
    accuracy = float(np.mean(np.array(all_pred) == np.array(all_true))) if len(all_true) > 0 else 0.0
    macro_f1 = f1_score(all_true, all_pred, average='macro', zero_division=0)
    
    return avg_loss, accuracy, macro_f1, all_pred, all_true


def print_class_counts(windows: list, title: str, class_names: list):
    """Print class distribution."""
    y = np.array([w['y'] for w in windows], dtype=np.int64)
    counts = np.bincount(y, minlength=len(class_names))
    msg = ', '.join([f"{class_names[i]}:{int(counts[i])}" for i in range(len(class_names))])
    print(f'{title}: {msg}')

print('Dataset and training utilities ready.')

In [ ]:
# ===================== LOSO Training Loop =====================

def train_loso_fold(fold_idx: int, test_subject: str, train_subjects: list, val_subjects: list):
    """
    Train a single LOSO fold.
    
    Args:
        fold_idx: Fold number (1-indexed)
        test_subject: Subject to test on
        train_subjects: List of subjects for training
        val_subjects: List of subjects for validation
    
    Returns:
        Dictionary with fold metrics
    """
    print(f'\n{"="*80}')
    print(f'LOSO Fold {fold_idx}: Test={test_subject}')
    print(f'{"="*80}')
    
    # Build windows
    print(f'Building training windows from {len(train_subjects)} subjects...')
    train_windows = build_split_windows(train_subjects, is_train=True)
    
    print(f'Building validation windows from {len(val_subjects)} subjects...')
    val_windows = build_split_windows(val_subjects, is_train=False)
    
    print(f'Building test windows from {test_subject}...')
    test_windows = build_split_windows([test_subject], is_train=False)
    
    print(f'Windows: train={len(train_windows)}, val={len(val_windows)}, test={len(test_windows)}')
    print_class_counts(train_windows, 'Train', CLASS_NAMES)
    print_class_counts(val_windows, 'Val', CLASS_NAMES)
    print_class_counts(test_windows, 'Test', CLASS_NAMES)
    
    if len(train_windows) == 0 or len(val_windows) == 0 or len(test_windows) == 0:
        print(f'⚠ Fold {fold_idx} skipped (empty split)')
        return None
    
    # Create datasets and loaders
    train_ds = ACCDataset(train_windows)
    val_ds = ACCDataset(val_windows)
    test_ds = ACCDataset(test_windows)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Model initialization
    model = Mamba2Stress(
        n_classes=len(CLASS_NAMES),
        dropout=DROPOUT,
        d_model=MAMBA_D_MODEL,
        d_state=MAMBA_D_STATE,
        expand=MAMBA_EXPAND,
    ).to(DEVICE)
    
    # Loss and optimizer
    class_w = class_weights_from_windows(train_windows, n_classes=len(CLASS_NAMES)).to(DEVICE)
    print(f'Class weights: {[round(x, 4) for x in class_w.tolist()]}')
    
    criterion = nn.CrossEntropyLoss(weight=class_w)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Training loop
    best_val_f1 = 0.0
    best_state = None
    patience_counter = 0
    
    print(f'\nEpoch | TrainLoss TrainAcc | ValLoss ValAcc ValF1 | ES')
    print('-' * 70)
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_pred = []
        train_true = []
        
        for acc, y in train_loader:
            acc = acc.to(DEVICE)
            y = y.to(DEVICE)
            
            optimizer.zero_grad()
            logits = model(acc)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item() * y.size(0)
            train_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
            train_true.extend(y.detach().cpu().numpy().tolist())
        
        scheduler.step()
        
        # Validation
        val_loss, val_acc, val_f1, _, _ = eval_model(model, val_loader, criterion, DEVICE)
        train_acc = float(np.mean(np.array(train_pred) == np.array(train_true)))
        train_loss_avg = train_loss / max(len(train_true), 1)
        
        # Early stopping
        improved = val_f1 > (best_val_f1 + EARLY_STOP_F1_MIN_DELTA)
        if improved:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            mark = ' *'
        else:
            patience_counter += 1
            mark = ''
        
        print(f'{epoch:5d} | {train_loss_avg:8.4f} {train_acc:8.2%} | {val_loss:7.4f} {val_acc:7.2%} {val_f1:6.3f} | {patience_counter:2d}/{EARLY_STOP_PATIENCE}{mark}')
        
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping at epoch {epoch} (best val F1: {best_val_f1:.3f})')
            break
    
    # Load best model and evaluate on test
    if best_state is not None:
        model.load_state_dict(best_state)
    
    test_loss, test_acc, test_f1, test_pred, test_true = eval_model(model, test_loader, criterion, DEVICE)
    
    print(f'\nTest Results: Loss={test_loss:.4f}, Acc={test_acc:.2%}, F1={test_f1:.3f}')
    print(f'Classification Report (Test):')
    print(classification_report(test_true, test_pred, target_names=CLASS_NAMES, zero_division=0))
    
    cm = confusion_matrix(test_true, test_pred, labels=[0, 1])
    print(f'Confusion Matrix (Test):')
    print(cm)
    
    # Save checkpoint
    ckpt_path = OUTPUT_DIR / f'mamba2_fold_{fold_idx:03d}.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'fold_idx': fold_idx,
        'test_subject': test_subject,
        'class_names': CLASS_NAMES,
    }, ckpt_path)
    
    fold_result = {
        'fold_idx': fold_idx,
        'test_subject': test_subject,
        'test_f1': test_f1,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'best_val_f1': best_val_f1,
        'n_train': len(train_windows),
        'n_val': len(val_windows),
        'n_test': len(test_windows),
    }
    
    return fold_result

print('LOSO training function ready.')

In [ ]:
# ===================== Run Full LOSO Training =====================

print(f'\n{"#"*80}')
print(f'# Running Full LOSO Cross-Validation')
print(f'# Total folds: {len(all_subjects)}')
print(f'{"#"*80}')

loso_results = []

for fold_idx, test_subject in enumerate(all_subjects, 1):
    # Get LOSO split
    train_subjects, val_subjects, test_subjects = load_loso_fold(test_subject, all_subjects, seed=SEED + fold_idx)
    
    # Train fold
    fold_result = train_loso_fold(fold_idx, test_subject, train_subjects, val_subjects)
    
    if fold_result is not None:
        loso_results.append(fold_result)
    
    # Quick break for testing (comment out for full run)
    # if fold_idx >= 3:
    #     print('\n[DEBUG] Breaking after 3 folds for testing')
    #     break

print(f'\n{"#"*80}')
print(f'# LOSO Training Complete')
print(f'{"#"*80}')

# Summarize results
results_df = pd.DataFrame(loso_results)
print(f'\nLOSO Summary:')
print(f'Completed folds: {len(results_df)}/{len(all_subjects)}')
print(f'\nPer-fold F1 scores:')
print(results_df[['fold_idx', 'test_subject', 'test_f1', 'test_acc']].to_string(index=False))

print(f'\nMacro-averaged metrics:')
print(f'  F1:  {results_df["test_f1"].mean():.4f} ± {results_df["test_f1"].std():.4f}')
print(f'  Acc: {results_df["test_acc"].mean():.4f} ± {results_df["test_acc"].std():.4f}')

# Save results
results_csv = OUTPUT_DIR / 'loso_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'\nResults saved to: {results_csv}')

In [ ]:
# ===================== EmpaticaE4 Data Loading =====================

def load_empatica_subject(subject_dir: Path) -> np.ndarray:
    """Load ACC from EmpaticaE4 subject (flat structure)."""
    acc_path = subject_dir / 'ACC.csv'
    
    if not acc_path.exists():
        return None
    
    acc = read_csv_array(acc_path)
    
    # Ensure 3-axis format
    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)
    if acc.shape[1] < 3:
        acc = np.tile(acc, (1, 3))[:, :3]
    else:
        acc = acc[:, :3]
    
    return acc.astype(np.float32)

def create_windows_acc_nolabel(acc: np.ndarray) -> list:
    """Create non-overlapping windows from ACC (for testing)."""
    window_samples = LEN_ACC
    windows = []
    
    for start in range(0, len(acc) - window_samples + 1, window_samples):
        acc_win = acc[start:start + window_samples].T.copy()  # (3, 1920)
        windows.append({
            'acc': acc_win.astype(np.float32),
        })
    
    return windows

def compute_catsa_global_stats(train_subjects: list) -> dict:
    """Compute global normalization stats from CATSA training subjects."""
    all_acc = []
    
    for subject in train_subjects:
        sdir = DATASET_ROOT / subject
        acc = load_acc_only(sdir, 'Baseline')
        if acc is not None:
            all_acc.append(acc)
    
    if not all_acc:
        return None
    
    acc_all = np.vstack(all_acc)
    acc_mean = acc_all.mean(axis=0, keepdims=True)
    acc_std = acc_all.std(axis=0, keepdims=True) + 1e-6
    
    return {
        'acc_mean': acc_mean.astype(np.float32),
        'acc_std': acc_std.astype(np.float32),
    }

def predict_on_empatica(models_dir: Path, empatica_dir: Path, global_stats: dict):
    """
    Evaluate ensemble on EmpaticaE4 subjects.
    
    Args:
        models_dir: Directory containing fold checkpoints
        empatica_dir: EmpaticaE4 subjects directory
        global_stats: Global normalization stats from CATSA training
    
    Returns:
        Dictionary with per-subject and ensemble results
    """
    # Load all checkpoints
    ckpt_paths = sorted(models_dir.glob('mamba2_fold_*.pth'))
    models = []
    
    for ckpt_path in ckpt_paths:
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model = Mamba2Stress(
            n_classes=len(CLASS_NAMES),
            dropout=0.0,  # No dropout for inference
            d_model=MAMBA_D_MODEL,
            d_state=MAMBA_D_STATE,
            expand=MAMBA_EXPAND,
        ).to(DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        models.append(model)
    
    print(f'Loaded {len(models)} fold models for ensemble')
    
    # Get EmpaticaE4 subjects
    empatica_subjects = sorted([p.name for p in empatica_dir.glob('subject_*') if p.is_dir()])
    print(f'Found {len(empatica_subjects)} EmpaticaE4 subjects')
    
    # Evaluate each subject
    results = {
        'subjects': [],
        'ensemble_preds': [],
        'ensemble_probs': [],
        'fold_predictions': defaultdict(list),
    }
    
    for subject_name in empatica_subjects:
        subject_dir = empatica_dir / subject_name
        acc = load_empatica_subject(subject_dir)
        
        if acc is None:
            print(f'  ⚠ {subject_name}: ACC not found')
            continue
        
        print(f'  Processing {subject_name} (ACC shape: {acc.shape})')
        
        # Apply global normalization
        acc_norm = (acc - global_stats['acc_mean']) / global_stats['acc_std']
        
        # Create windows
        windows = create_windows_acc_nolabel(acc_norm)
        
        if not windows:
            print(f'    ⚠ No windows created')
            continue
        
        # Ensemble prediction
        all_probs = []
        
        for fold_idx, model in enumerate(models):
            fold_preds = []
            
            for window in windows:
                acc_win = torch.tensor(window['acc'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                
                with torch.no_grad():
                    logits = model(acc_win)
                    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                
                fold_preds.append(probs)
            
            fold_preds = np.array(fold_preds)  # (n_windows, 2)
            all_probs.append(fold_preds.mean(axis=0))  # average over windows
        
        # Ensemble average across folds
        ensemble_prob = np.array(all_probs).mean(axis=0)  # (2,)
        ensemble_pred = ensemble_prob.argmax()
        
        results['subjects'].append(subject_name)
        results['ensemble_preds'].append(ensemble_pred)
        results['ensemble_probs'].append(ensemble_prob)
        
        print(f'    Ensemble: Pred={CLASS_NAMES[ensemble_pred]}, Prob=[{ensemble_prob[0]:.3f}, {ensemble_prob[1]:.3f}]')
    
    return results

print('EmpaticaE4 evaluation functions ready.')

In [ ]:
# ===================== Run EmpaticaE4 Evaluation =====================

print(f'\n{"#"*80}')
print(f'# Evaluating on EmpaticaE4 Stress Dataset')
print(f'{"#"*80}')

# Compute global normalization stats from all CATSA training subjects
print('\nComputing global normalization stats from CATSA...')
catsa_train_subjects = [s for s in all_subjects if s != all_subjects[0]]  # Use most subjects as reference
global_stats = compute_catsa_global_stats(catsa_train_subjects)

if global_stats is None:
    print('⚠ Failed to compute global stats')
else:
    print(f'Global stats computed:')
    print(f'  ACC mean: {global_stats["acc_mean"].flatten()}')
    print(f'  ACC std:  {global_stats["acc_std"].flatten()}')
    
    # Evaluate on EmpaticaE4
    empatica_results = predict_on_empatica(OUTPUT_DIR, EMPATICA_ROOT, global_stats)
    
    # Save results
    empatica_df = pd.DataFrame({
        'subject': empatica_results['subjects'],
        'prediction': [CLASS_NAMES[p] for p in empatica_results['ensemble_preds']],
        'pred_idx': empatica_results['ensemble_preds'],
        'prob_nonstress': [p[0] for p in empatica_results['ensemble_probs']],
        'prob_stress': [p[1] for p in empatica_results['ensemble_probs']],
    })
    
    empatica_csv = OUTPUT_DIR / 'empatica_predictions.csv'
    empatica_df.to_csv(empatica_csv, index=False)
    
    print(f'\n{"="*80}')
    print(f'EmpaticaE4 Prediction Summary:')
    print(f'{"="*80}')
    print(empatica_df.to_string(index=False))
    
    print(f'\nPredictions saved to: {empatica_csv}')
    print(f'\nStress distribution:')
    stress_counts = empatica_df['prediction'].value_counts()
    for label in CLASS_NAMES:
        count = stress_counts.get(label, 0)
        pct = 100 * count / len(empatica_df)
        print(f'  {label}: {count}/{len(empatica_df)} ({pct:.1f}%)')

print(f'\n{"#"*80}')
print(f'# Evaluation Complete')
print(f'{"#"*80}')

In [ ]:
# ===================== Final Summary =====================

print(f'\n{"#"*80}')
print(f'# FINAL SUMMARY - Mamba2 ACC-Only LOSO Training')
print(f'{"#"*80}')

print(f'\nConfiguration:')
print(f'  Model: Mamba2 (d_model=3, d_state=64, expand=2)')
print(f'  Input: ACC only (32Hz, 3-axis)')
print(f'  Window: 60s @ 32Hz (1920 samples) → downsampled to 240 for Mamba2')
print(f'  Training stride: 20s')
print(f'  Eval stride: 60s')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Epochs: {EPOCHS}')
print(f'  Learning rate: {LR}')
print(f'  Early stopping: patience={EARLY_STOP_PATIENCE}, min_delta={EARLY_STOP_F1_MIN_DELTA}')
print(f'  Device: {DEVICE}')

print(f'\nLOSO Results (CATSA):')
if len(results_df) > 0:
    print(f'  Total folds: {len(results_df)}')
    print(f'  Mean F1:  {results_df["test_f1"].mean():.4f} ± {results_df["test_f1"].std():.4f}')
    print(f'  Mean Acc: {results_df["test_acc"].mean():.4f} ± {results_df["test_acc"].std():.4f}')
    print(f'  Min F1:   {results_df["test_f1"].min():.4f}')
    print(f'  Max F1:   {results_df["test_f1"].max():.4f}')
    
    worst_fold = results_df.loc[results_df['test_f1'].idxmin()]
    best_fold = results_df.loc[results_df['test_f1'].idxmax()]
    print(f'  Worst fold: {worst_fold["test_subject"]} (F1={worst_fold["test_f1"]:.4f})')
    print(f'  Best fold:  {best_fold["test_subject"]} (F1={best_fold["test_f1"]:.4f})')

print(f'\nEmpaticaE4 Results:')
if 'empatica_results' in locals() and len(empatica_df) > 0:
    print(f'  Total subjects: {len(empatica_df)}')
    print(f'  Stress predictions: {(empatica_df["prediction"] == "Stress").sum()}/{len(empatica_df)}')
    print(f'  NonStress predictions: {(empatica_df["prediction"] == "NonStress").sum()}/{len(empatica_df)}')
    print(f'  Mean stress probability: {empatica_df["prob_stress"].mean():.3f}')

print(f'\nOutput Files:')
print(f'  Results dir: {OUTPUT_DIR}')
print(f'  LOSO results: {OUTPUT_DIR / "loso_results.csv"}')
print(f'  Model checkpoints: {OUTPUT_DIR / "mamba2_fold_*.pth"}')
if 'empatica_df' in locals():
    print(f'  EmpaticaE4 predictions: {OUTPUT_DIR / "empatica_predictions.csv"}')

print(f'\n{"#"*80}')
print(f'# Training Complete!')
print(f'{"#"*80}')